In [1]:
from matplotlib import pyplot as plt
from serial import SerialException
import enum
import numpy as np

from discovery import Discovery
from driver import Driver
from load_pickle import load

try:
    Encryption_Board = Driver()
    Oscilloscope = Discovery()
except SerialException:
    print("Encryption board not found. Please check the connection.")
except SystemExit:
    print("Oscilloscope not found. Please check the connection.")

# Setup reading channels for the oscilloscope
class CHANNELS(enum.Enum):
    POWER_CONSUMPTION = 0
    SBOX_ACCESS_TRIGGER = 1
    ENCRYPTION_START_TRIGGER = 2
    CH3 = 3
    CH4 = 4

Encryption board not found. Please check the connection.


In [2]:
def generateRandomPlaintext() -> int:
    return np.random.randint(0, 2**128)

def plotCurrentWaveform(measure: np.ndarray):
    plt.plot(measure)
    plt.xlabel("Time (ms)")
    plt.ylabel("Current (mA)")
    plt.title("Current waveform")
    plt.show()

## Board Functions

In [3]:
def encryptPlaintext(plaintext: int) -> int:
    ciphertext = Encryption_Board.encrypt(plaintext)
    return ciphertext

## Oscilloscope functions

In [4]:
def measureCurrent(measure: np.ndarray, rise: int, fall: int) -> float:
    """
    Digital pin 5:
        Rise: begin of 1st stage sbox calculation
        Fall: end of 1st stage sbox calculation
    """
    return measure[rise:fall].avg()

def prepareOscilloscope(channel: CHANNELS = CHANNELS.SBOX_ACCESS_TRIGGER):
    """
    Activates the specified channel as trigger to the readings
    if no <channel> is specified, the default is SBOX_ACCESS_TRIGGER
    """
    if channel is CHANNELS.SBOX_ACCESS_TRIGGER:
        Oscilloscope.configure() # Activate channel 0 and 1
    else:
        Oscilloscope.enableChannel(0)
        Oscilloscope.setMode('normal')
        Oscilloscope.enableChannel(channel.value)
        Oscilloscope.setOffset(channel.value, 0.0)
        Oscilloscope.setTriggerEdge(channel.value, 1, "rising")
    
    # Start the oscilloscope to be ready for readings
    Oscilloscope.run()

def waitForAcquisitionDone():
    while not Oscilloscope.isReady():
        pass

def getOscilloscopeReadings(channel: CHANNELS=CHANNELS.POWER_CONSUMPTION) -> np.ndarray:
    return np.array(
        Oscilloscope.getSamples(channel.value)
    )

def measurePowerTraces(num: int) -> tuple[list[int], np.ndarray]:
    T: list[int] = []
    M: np.ndarray = np.zeros((num, 4000)) # 4000 samples per trace (this number was taken from the provided traces)
    for i in range(num):
        plaintext = generateRandomPlaintext()
        T.append(plaintext)

        prepareOscilloscope(CHANNELS.SBOX_ACCESS_TRIGGER)
        ciphertext = Encryption_Board.encrypt(plaintext)
        waitForAcquisitionDone()
        M[i] = getOscilloscopeReadings(CHANNELS.POWER_CONSUMPTION)
    Oscilloscope.close()
    return T, M

# CPA Attack

In [5]:
def hamming_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Calculate the Hamming distance between two arrays of binary strings.
    A XOR B, then convert to bits and sum the number of 1s in each row
    """
    xor = np.bitwise_xor(a, b)
    bits_representation = np.unpackbits(xor, axis=1)
    hamming_distance = bits_representation.sum(axis=1)
    return hamming_distance.reshape(-1, 1) # Convert to column vector

# Attack using provided powertraces

In [6]:
P: list[int]
M: np.ndarray
P, M = load("./testsca.pickle")

In [7]:
print(len(P), M.shape)
print(P[0])

500 (500, 4000)
8178686806597379442190969825167098279


## CPA Implementation

In [8]:
from aes_sbox import SBOX

# Precompute helpers for the leakage model (HW of S-box output).
SBOX_ARRAY = np.array(SBOX, dtype=np.uint8)

def extract_plaintext_bytes(plaintexts: list[int]) -> np.ndarray:
    pt_bytes = np.zeros((len(plaintexts), 16), dtype=np.uint8)
    for i, p in enumerate(plaintexts):
        for b in range(16):
            shift = 8 * (15 - b)
            pt_bytes[i, b] = (p >> shift) & 0xFF
    return pt_bytes

def cpa_byte(pt_bytes: np.ndarray, traces: np.ndarray, byte_index: int) -> dict[int, float]:
    N, _ = traces.shape
    plaintexts_byte = pt_bytes[:, byte_index].reshape(-1, 1)  # (N, 1)


    sum_W  = traces.sum(axis=0)
    sum_W2 = (traces ** 2).sum(axis=0)

    # Yeah, list is enough as its a number (instead of dict),
    # but this is more intuitive
    correlations: dict[int, float] = {}
    for key_guess in range(256):
        key_guess_bytes = np.uint8(key_guess)
        R = np.full((N, 1), key_guess, dtype=np.uint8)
        sbox_out = SBOX_ARRAY[plaintexts_byte.flatten() ^ key_guess].reshape(-1, 1)
        H = hamming_distance(
            sbox_out, R
        )

        sum_H  = H.sum()           # scalar
        sum_H2 = (H ** 2).sum()    # scalar
        sum_WH = (traces * H).sum(axis=0)  # (T,) — H broadcasts over T
        
        numerator   = N * sum_WH - sum_W * sum_H          # (T,)
        first_term  = N * sum_W2 - sum_W ** 2             # (T,)
        second_term = N * sum_H2 - sum_H ** 2             # scalar

        numerator = N * np.sum(traces * H, axis=0) - np.sum(traces, axis=0) * np.sum(H, axis=0)
        # Denominator
        first_term = (N * np.sum(traces * traces, axis=0) - np.sum(traces, axis=0) ** 2)
        second_term = (N * np.sum(H * H, axis=0) - np.sum(H, axis=0) ** 2)
        denom = np.sqrt(np.maximum(first_term, 0)) * np.sqrt(max(second_term, 0))

        with np.errstate(invalid='ignore', divide='ignore'):
            corr = np.where(denom != 0, numerator / denom, 0.0)  # (T,)

        correlations[key_guess] = float(np.max(np.abs(corr)))
    return correlations

def cpa_attack(plaintexts: list[int], traces: np.ndarray) -> tuple[list[int], list[float]]:
    pt_bytes = extract_plaintext_bytes(plaintexts)
    key_bytes: list[int] = []
    key_scores: list[float] = []
    for b in range(16):
        correlations = cpa_byte(pt_bytes, traces, b)
        guess = max(correlations, key=lambda key_guess: correlations[key_guess])
        print(f"Byte {b}: Guess = {guess:02x}, Correlation = {correlations[guess]:.4f}")
        score = correlations[guess]
        key_bytes.append(guess)
        key_scores.append(score)
    return key_bytes, key_scores

key_bytes, key_scores = cpa_attack(P, M)
key_hex = "".join(f"{b:02x}" for b in key_bytes)
print(f"Recovered key: {key_hex}")

Byte 0: Guess = 75, Correlation = 0.3911
Byte 1: Guess = e8, Correlation = 0.3547
Byte 2: Guess = cc, Correlation = 0.3395
Byte 3: Guess = 74, Correlation = 0.4480
Byte 4: Guess = 97, Correlation = 0.4710
Byte 5: Guess = f7, Correlation = 0.7911
Byte 6: Guess = 46, Correlation = 0.6651
Byte 7: Guess = a9, Correlation = 0.3444
Byte 8: Guess = e4, Correlation = 0.3030
Byte 9: Guess = 76, Correlation = 0.3523
Byte 10: Guess = 80, Correlation = 0.7014
Byte 11: Guess = 62, Correlation = 0.2560
Byte 12: Guess = 71, Correlation = 0.5529
Byte 13: Guess = 84, Correlation = 0.4779
Byte 14: Guess = 17, Correlation = 0.2984
Byte 15: Guess = e4, Correlation = 0.3460
Recovered key: 75e8cc7497f746a9e4768062718417e4


In [ ]:
import aes_encrypt
aes_encrypt.encrypt(0x0, int(key_hex, 16)).to_bytes(16, 'big').hex()

'8cd8d4ec031a8e5ccf04a2f13764dcb7'

In [10]:
print()